In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import json
import hashlib
import datetime
import os

base_dir = r"D:\Machine Learning Dataset\unsw nb15\archive"
output_dir = r"D:\Machine Learning Dataset"

train_path = os.path.join(base_dir, "UNSW_NB15_testing-set.csv")  
test_path = os.path.join(base_dir, "UNSW_NB15_training-set.csv") 

# 模型训练与基准测试

print("正在载入数据...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

drop_cols = ['id', 'attack_cat', 'label']
y_train = train_df['label']
y_test = test_df['label']

X_train_raw = train_df.drop(drop_cols, axis=1)
X_test_raw = test_df.drop(drop_cols, axis=1)

print("进行特征工程...")
X_train_encoded = pd.get_dummies(X_train_raw)
X_test_encoded = pd.get_dummies(X_test_raw)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_encoded)
X_test_final = scaler.transform(X_test_encoded)

print("训练数字取证分析引擎 (Random Forest)...")
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)
print(f"完成！当前取证准确率: {model.score(X_test_final, y_test):.4f}")

# Explainable AI)

print("\n正在生成并保存tezhengtu ...")
plt.figure(figsize=(10, 6))
feature_names = X_train_encoded.columns
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:10]

sns.barplot(x=importances[indices], 
            y=[feature_names[i] for i in indices], 
            hue=[feature_names[i] for i in indices],
            palette="viridis", 
            legend=False)
plt.title('Top 10 Forensic Evidence Features (Explainability)')
plt.xlabel('Importance Score')
plt.tight_layout()

image_path = os.path.join(output_dir, "Feature_Importance_Plot.png")
plt.savefig(image_path, dpi=300)
print(f"图片已保存至: {image_path}")

# 取证，提取证据并生成JSON
print("\n" + "="*50)


# 找到所有被预测为 1 的数据行
anomaly_indices = (y_pred == 1)
evidence_records = test_df[anomaly_indices]
total_evidence_count = len(evidence_records)

print(f"发现 {total_evidence_count} 条攻击记录，正在打包...")

evidence_list = []
for index, row in evidence_records.iterrows():
    
    protocol = str(row.get('proto', 'UNKNOWN'))
    service = str(row.get('service', 'UNKNOWN'))
    state = str(row.get('state', 'UNKNOWN'))
    
    evidence_item = {
        "Evidence_ID": f"EVID-{index}",
        "Timestamp": str(datetime.datetime.utcnow()),
        "Searchable_Keywords": [f"PROTOCOL:{protocol}", f"SERVICE:{service}", f"STATE:{state}"],
        "Forensic_Metrics": {
            "Protocol": protocol,
            "Service": service,
            "Connection_State": state,
            "Source_Bytes": int(row.get('sbytes', 0)),
            "Destination_Bytes": int(row.get('dbytes', 0)),
            "Time_To_Live": int(row.get('sttl', 0))
        }
    }
    
    # 计算区块链哈希
    evidence_string = json.dumps(evidence_item, sort_keys=True)
    blockchain_hash = hashlib.sha256(evidence_string.encode('utf-8')).hexdigest()
    evidence_item["Blockchain_SHA256_Hash"] = blockchain_hash
    
    evidence_list.append(evidence_item)

# 将证据写入 JSON 文件
json_path = os.path.join(output_dir, "Forensic_Evidence_Report.json")
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(evidence_list, f, indent=4)

print("\n" + "="*50)
print(f"成功提取并封装了 {total_evidence_count} 份完整的数字证据！")
print(f"JSON 证据已保存至: {json_path}")
print("="*50)